In [0]:
from pyspark.sql.functions import col, count, round, sum, avg, when

orders = spark.table("silver.olist.orders")
order_items = spark.table("silver.olist.order_items")
order_payments = spark.table("silver.olist.order_payments")
customers = spark.table("silver.olist.customers")

# Total payment per order
payments_agg = order_payments.groupBy("order_id") \
    .agg(round(sum("payment_value"), 2).alias("total_payment_value"))

# Items per order
items_agg = order_items.groupBy("order_id") \
    .agg(count("order_item_id").alias("items_in_order"))

# Join
df = orders \
    .join(customers.select("customer_id", "customer_unique_id", 
                           "customer_state", "customer_city"), 
          on="customer_id", how="left") \
    .join(payments_agg, on="order_id", how="left") \
    .join(items_agg, on="order_id", how="left")

# Aggregate per customer profile
gold_customers = df.groupBy("customer_unique_id", "customer_state", "customer_city") \
    .agg(
        count("order_id").alias("total_orders"),
        round(sum("total_payment_value"), 2).alias("total_spent"),
        round(avg("total_payment_value"), 2).alias("avg_order_value"),
        round(avg("items_in_order"), 1).alias("avg_items_per_order")
    ) \
    .withColumn(
        "customer_segment",
        when(col("total_orders") >= 3, "Loyal")
        .when(col("total_orders") == 2, "Returning")
        .otherwise("One-time")
    ) \
    .orderBy(col("total_spent").desc())

# Save to Gold Layer
gold_customers.write.format("delta").mode("overwrite") \
    .option("path", "abfss://gold@project1storagek.dfs.core.windows.net/customer_orders") \
    .saveAsTable("gold.olist.customer_orders")

print("customer_orders done:", gold_customers.count())
gold_customers.show(10)

customer_orders done: 96219
+--------------------+--------------+--------------+------------+-----------+---------------+-------------------+----------------+
|  customer_unique_id|customer_state| customer_city|total_orders|total_spent|avg_order_value|avg_items_per_order|customer_segment|
+--------------------+--------------+--------------+------------+-----------+---------------+-------------------+----------------+
|0a0a92112bd4c708c...|            RJ|rio de janeiro|           1|   13664.08|       13664.08|                8.0|        One-time|
|46450c74a0d8c5ca9...|            SC| florianopolis|           3|    9553.02|        3184.34|                1.0|           Loyal|
|da122df9eeddfedc1...|            RJ|      araruama|           2|    7571.63|        3785.82|                1.0|       Returning|
|763c8b1c9c68a0229...|            ES|    vila velha|           1|    7274.88|        7274.88|                4.0|        One-time|
|dc4802a71eae9be1d...|            MS|  campo grande|   

In [0]:
spark.sql("DROP TABLE IF EXISTS gold.olist.customer_orders")

DataFrame[]